In [4]:
import json

In [2]:
import nconfig

In [ ]:
import aiofiles

In [22]:
chomps_aws_table_file = nconfig.PROJECT_DIR / "outputs" / "chomps_aws_result.json"

In [23]:
chomps_aws_table_data = ""

with open(chomps_aws_table_file, 'r') as f:
    chomps_aws_table_data = f.read()

In [24]:
chomps_json_data = json.loads(chomps_aws_table_data)

In [25]:
chomps_json_data

{'DocumentMetadata': {'Pages': 4},
 'JobStatus': 'SUCCEEDED',
 'NextToken': 'cZd5x8b4o0ellSHHIiKpgc8gPzerPNofCmCLFEUnupOMZFJ4f+ItuQecsf6itNQk1gUcpCxucqy7+ib/RopTxSXeoQ2zzLx2xsgU4eaTUt1oc4OLPKb/+OkoIZ3RVwQ691Y7/DE=',
 'Blocks': [{'BlockType': 'PAGE',
   'Geometry': {'BoundingBox': {'Width': 0.9999957084655762,
     'Height': 1.0,
     'Left': 4.268024440534646e-06,
     'Top': 0.0},
    'Polygon': [{'X': 4.64421782453428e-06, 'Y': 3.8718553696526214e-06},
     {'X': 1.0, 'Y': 0.0},
     {'X': 1.0, 'Y': 1.0},
     {'X': 4.268024440534646e-06, 'Y': 1.0}]},
   'Id': '35b69625-a7f9-4f6f-8579-4a99f482f699',
   'Relationships': [{'Type': 'CHILD',
     'Ids': ['39a33987-5c18-4f27-900d-b6ea3676df3f',
      '36ab80a3-445b-44e2-9145-6cc4af91ab46',
      '9f22d480-b996-4d62-822c-c75f9c75f531',
      'aab47be5-6f0c-4003-b2c6-204618e3bc86',
      '874bb96b-2d11-4d17-aee7-2d586b7d523c',
      '6fb1e965-d842-4beb-95d5-735caadbf344',
      '21c82c06-8a43-4902-911d-a7585f300aa6',
      'd732f40f-758b-40

In [15]:

output = []

# Define scoring columns and labels
score_columns = {
    2: "YES",
    3: "SOMETIMES",
    4: "NOT YET"
}

# Iterate through all tables and rows
for table in chomps_json_data["tables"]:
    for row in table["rows"]:
        cells = {cell["column_index"]: cell["text"] for cell in row["cells"]}
        obs_text = cells.get(1, "").strip()

        # Skip headers or empty rows
        if not obs_text or "my child" in obs_text.lower():
            continue

        # Find which score column is marked (in real case: find SelectionStatus == SELECTED)
        selected_score = None
        for col_idx, label in score_columns.items():
            text = cells.get(col_idx, "").strip()
            if "SELECTED" in text or "✓" in text or "☑" in text:  # If symbol present
                selected_score = (label, int(cells[2])) if col_idx == 2 else \
                                 (label, int(cells[3])) if col_idx == 3 else \
                                 (label, int(cells[4]))
                break

        if selected_score:
            output.append({
                "observation": obs_text,
                "score_label": selected_score[0],
                "score_value": selected_score[1]
            })

# Print result
print(json.dumps(output, indent=2))

[]


In [26]:
blocks = chomps_json_data["Blocks"]

cell_map = {}
selected_map = {}
text_map = {}

# First pass: map block IDs to text and structure
for block in blocks:
    if block["BlockType"] == "CELL":
        row = block.get("RowIndex")
        col = block.get("ColumnIndex")
        block_id = block["Id"]
        cell_map[(row, col)] = block
        if "Relationships" in block:
            for rel in block["Relationships"]:
                if rel["Type"] == "CHILD":
                    for child_id in rel["Ids"]:
                        text_map[child_id] = (row, col, block_id)
    elif block["BlockType"] == "SELECTION_ELEMENT" and block.get("SelectionStatus") == "SELECTED":
        selected_map[block["Id"]] = block

# Second pass: resolve selected checkboxes to cell positions
selected_cells = {}
for sel_id in selected_map:
    if sel_id in text_map:
        row, col, parent_id = text_map[sel_id]
        selected_cells[row] = col  # One selected per row

# Now map row to observation and score
output = []
score_labels = {2: ("YES", 2), 3: ("SOMETIMES", 1), 4: ("NOT YET", 0)}

for (row, col), block in cell_map.items():
    if col == 1:  # Observation column
        observation = block.get("Text", "").strip()
        selected_col = selected_cells.get(row)
        if selected_col in score_labels:
            label, score = score_labels[selected_col]
            output.append({
                "observation": observation,
                "score_label": label,
                "score_value": score
            })

# Output the structured result
with open("mapped_scores.json", "w") as f_out:
    json.dump(output, f_out, indent=2)

print(f"✅ Extracted {len(output)} mapped observation-score pairs.")


✅ Extracted 15 mapped observation-score pairs.
